In [1]:
# loading libraries
import numpy as np

import matplotlib.pyplot as plt 
import seaborn as sns

import pandas as pd

In [2]:
# import functions
import coriolis_functions 
# import selfbuild module
import coriolis_module
# check out if import worked as expected
print(dir(coriolis_module))
print(dir(coriolis_functions))

['__doc__', '__file__', '__loader__', '__name__', '__package__', '__spec__', 'calculate_drift_distances', 'calculate_velocities', 'coriolis_acc', 'direction_vector', 'radius_at_latitude', 'rotation_matrix']
['__builtins__', '__cached__', '__doc__', '__file__', '__loader__', '__name__', '__package__', '__spec__', 'calculate_drift_distances', 'calculate_total_drift', 'calculate_velocities', 'coriolis_acc', 'haversine', 'main', 'np', 'radius_at_latitude', 'rotation_matrix']


In [3]:
fdf = pd.read_csv("data/flights_sample_3m.csv")
ldf = pd.read_csv("data/airports.csv")

print(len(fdf), " : number of flights in dataframe")
print(len(ldf), " : number of airport-locations in dataframe")

print(fdf.columns)

3000000  : number of flights in dataframe
341  : number of airport-locations in dataframe
Index(['FL_DATE', 'AIRLINE', 'AIRLINE_DOT', 'AIRLINE_CODE', 'DOT_CODE',
       'FL_NUMBER', 'ORIGIN', 'ORIGIN_CITY', 'DEST', 'DEST_CITY',
       'CRS_DEP_TIME', 'DEP_TIME', 'DEP_DELAY', 'TAXI_OUT', 'WHEELS_OFF',
       'WHEELS_ON', 'TAXI_IN', 'CRS_ARR_TIME', 'ARR_TIME', 'ARR_DELAY',
       'CANCELLED', 'CANCELLATION_CODE', 'DIVERTED', 'CRS_ELAPSED_TIME',
       'ELAPSED_TIME', 'AIR_TIME', 'DISTANCE', 'DELAY_DUE_CARRIER',
       'DELAY_DUE_WEATHER', 'DELAY_DUE_NAS', 'DELAY_DUE_SECURITY',
       'DELAY_DUE_LATE_AIRCRAFT'],
      dtype='object')


In [4]:
fdf = fdf.drop( ['AIRLINE', 'AIRLINE_DOT', 'AIRLINE_CODE', 'DOT_CODE',
       'FL_NUMBER', 'ORIGIN_CITY', 'DEST_CITY',
       'CRS_DEP_TIME', 'DEP_TIME', 'DEP_DELAY', 'TAXI_OUT',
       'TAXI_IN', 'CRS_ARR_TIME', 'ARR_TIME', 'ARR_DELAY',
       'CANCELLATION_CODE', 'CRS_ELAPSED_TIME',
       'ELAPSED_TIME', 'AIR_TIME', 'DISTANCE', 'DELAY_DUE_CARRIER',
       'DELAY_DUE_WEATHER', 'DELAY_DUE_NAS', 'DELAY_DUE_SECURITY',
       'DELAY_DUE_LATE_AIRCRAFT'] ,axis=1)

In [5]:
fdf.dtypes

FL_DATE        object
ORIGIN         object
DEST           object
WHEELS_OFF    float64
WHEELS_ON     float64
CANCELLED     float64
DIVERTED      float64
dtype: object

In [6]:
# convert to datetime
fdf["FL_DATE"] = pd.to_datetime(fdf["FL_DATE"])

In [7]:
boolean_columns = ["CANCELLED", "DIVERTED"]
fdf[boolean_columns] = fdf[boolean_columns].astype(bool)

In [8]:
fdf = fdf[~fdf["CANCELLED"]]
fdf = fdf.drop(["CANCELLED"], axis=1)

In [9]:
# defining a helper function
def convert_time(df, time_columns):
    """ Convert integer time columns to HH:MM format """
    for col in time_columns:
        # Handle missing values and convert times
        df[col] = df[col].fillna(0).astype(int).apply(lambda x: f"{x//100:02d}:{x%100:02d}")
        # Adjust for hours == 24
        df[col] = df[col].replace("24:00", "00:00")
    return df

# applying the time conversion
fdf = convert_time(fdf, ["WHEELS_OFF", "WHEELS_ON"])

fdf["woff_datetime"] = pd.to_datetime(fdf["FL_DATE"].astype(str) + " " + fdf["WHEELS_OFF"])
fdf["won_datetime"] = pd.to_datetime(fdf["FL_DATE"].astype(str) + " " + fdf["WHEELS_ON"])

# dropping original time columns
fdf = fdf.drop(["WHEELS_OFF", "WHEELS_ON"], axis=1)

In [10]:
# checking unique values in both datasets
fdf_airports = set(fdf["ORIGIN"].unique()).union(set(fdf["DEST"].unique()))
ldf_airports = set(ldf["IATA"].unique())  

# finding missing airport codes in ldf
missing_airports = fdf_airports.difference(ldf_airports)

# dropping rows where "ORIGIN" or "DEST" are in missing_airports
fdf = fdf[~fdf["ORIGIN"].isin(missing_airports) & ~fdf["DEST"].isin(missing_airports)]

# merging fdf with ldf to add geographical data for ORIGIN and DEST
fdf = pd.merge(fdf, ldf[["IATA", "LATITUDE", "LONGITUDE"]], 
                      left_on="ORIGIN", right_on="IATA", how="left")

fdf = pd.merge(fdf, ldf[["IATA", "LATITUDE", "LONGITUDE"]], 
                      left_on="DEST", right_on="IATA", how="left", suffixes=("_ORIGIN", "_DEST"))

# dropping original origin and destination columns
fdf = fdf.drop(["ORIGIN", "DEST"], axis=1)

# enforcing categories on newly generated columns
fdf[["IATA_ORIGIN", "IATA_DEST"]] = fdf[["IATA_ORIGIN", "IATA_DEST"]].astype("category")

In [11]:
fdf["airtime"] = fdf["won_datetime"] - fdf["woff_datetime"]

In [12]:
# calculating haversinedistance
fdf["haversine_distance"] = coriolis_functions.haversine(
    fdf["LATITUDE_ORIGIN"],
    fdf["LONGITUDE_ORIGIN"],
    fdf["LATITUDE_DEST"], 
    fdf["LONGITUDE_DEST"]
)

In [13]:
fdf["average_velocity"] = fdf["haversine_distance"] / fdf["airtime"].dt.total_seconds()

In [14]:
# Apply direction vector function row-wise
directions = fdf.apply(
    lambda row: coriolis_module.direction_vector(
        row["LATITUDE_ORIGIN"], 
        row["LONGITUDE_ORIGIN"], 
        row["LATITUDE_DEST"], 
        row["LONGITUDE_DEST"]
    ), axis=1
)

# Split the resulting direction vectors into separate columns
fdf["x_direction"] = fdf["average_velocity"] * directions.apply(lambda x: x[0])
fdf["y_direction"] = fdf["average_velocity"] * directions.apply(lambda x: x[1])
fdf["z_direction"] = fdf["average_velocity"] * directions.apply(lambda x: x[2])

In [15]:
# checking dtypes, lost rows
print(fdf.dtypes, f"\n")
print(f"{len(fdf):,.2f},  : number of flights in dataframe\n")
fdf.head(10)

FL_DATE                datetime64[ns]
DIVERTED                         bool
woff_datetime          datetime64[ns]
won_datetime           datetime64[ns]
IATA_ORIGIN                  category
LATITUDE_ORIGIN               float64
LONGITUDE_ORIGIN              float64
IATA_DEST                    category
LATITUDE_DEST                 float64
LONGITUDE_DEST                float64
airtime               timedelta64[ns]
haversine_distance            float64
average_velocity              float64
x_direction                   float64
y_direction                   float64
z_direction                   float64
dtype: object 

2,870,784.00,  : number of flights in dataframe



,FL_DATE,DIVERTED,woff_datetime,won_datetime,IATA_ORIGIN,LATITUDE_ORIGIN,LONGITUDE_ORIGIN,IATA_DEST,LATITUDE_DEST,LONGITUDE_DEST,airtime,haversine_distance,average_velocity,x_direction,y_direction,z_direction
0,2019-01-09,False,2019-01-09 12:10:00,2019-01-09 14:43:00,FLL,26.072583,-80.152750,EWR,40.692497,-74.168661,0 days 02:33:00,1716.995370,0.187037,0.037057,0.108287,0.147931
1,2022-11-19,False,2022-11-19 21:23:00,2022-11-19 22:32:00,MSP,44.880547,-93.216922,SEA,47.448982,-122.309313,0 days 01:09:00,2245.634221,0.542424,-0.497692,0.210283,0.048030
2,2022-07-22,False,2022-07-22 10:20:00,2022-07-22 12:47:00,DEN,39.858408,-104.667002,MSP,44.880547,-93.216922,0 days 02:27:00,1092.568183,0.123874,0.111828,0.025427,0.046827
3,2023-03-06,False,2023-03-06 16:35:00,2023-03-06 18:44:00,MSP,44.880547,-93.216922,SFO,37.619002,-122.374843,0 days 02:09:00,2552.088033,0.329727,-0.318550,0.031904,-0.078918
4,2020-02-23,False,2020-02-23 18:53:00,2020-02-23 20:26:00,MCO,28.428889,-81.316028,DFW,32.895951,-97.037200,0 days 01:33:00,1581.828883,0.283482,-0.269764,0.041206,0.076755
5,2019-07-31,False,2019-07-31 12:52:00,2019-07-31 13:28:00,DAL,32.847114,-96.851772,OKC,35.393088,-97.600734,0 days 00:36:00,291.398713,0.134907,-0.022415,0.076955,0.108515
6,2023-06-11,False,2023-06-11 10:24:00,2023-06-11 11:22:00,DCA,38.852083,-77.037722,BOS,42.364348,-71.005179,0 days 00:58:00,641.567152,0.184358,0.120541,0.110422,0.085234
7,2019-07-08,False,2019-07-08 16:59:00,2019-07-08 19:27:00,HSV,34.640448,-86.773109,DCA,38.852083,-77.037722,0 days 02:28:00,985.086458,0.110933,0.092204,0.044896,0.042296
8,2023-02-12,False,2023-02-12 05:38:00,2023-02-12 06:58:00,IAH,29.980472,-95.339722,LAX,33.942536,-118.408074,0 days 01:20:00,2215.554211,0.461574,-0.419007,0.177088,0.078254
9,2020-08-22,False,2020-08-22 21:35:00,2020-08-22 23:53:00,SEA,47.448982,-122.309313,FAI,64.813677,-147.859669,0 days 02:18:00,2462.210595,0.297368,0.000853,0.267299,0.130303
